# Tableaux Post : moyenne par mois, différence essai 1 - essai 3 et pente sur les essais

À lancer APRÈS la création des tableaux bruts (`Summary_Table_Opto`, `Summary_Table_Chemo_Opto`) et la cellule "Compléter sex / implant" du notebook v7_2.

Crée dans le même dossier : `Summary_Table_Opto_Post_mean`, `Summary_Table_Opto_Post_delta`, `Summary_Table_Opto_Post_slope`, `Summary_Table_Opto_Chemo_Post_mean`, `Summary_Table_Opto_Chemo_Post_delta`, `Summary_Table_Opto_Chemo_Post_slope`, et les versions `_normalized` des tableaux **Post_mean** uniquement (le delta reste en valeurs brutes).

**Signe :** pour le delta (essai 1 - essai 3) et la pente, **positif = la valeur diminue au fil des essais** (pour `latency_to_reward_s` et `distance_to_reward_cm` : la souris s'améliore).


In [ ]:
##############################################################################
# Tableaux POST : moyenne par mois, différence essai 1 - essai 3, PENTE sur les essais, puis normalisation
#
# À partir des tableaux bruts (une ligne = un essai) on crée, pour Opto et Opto_Chemo :
#   Summary_Table_<Opto|Opto_Chemo>_Post_mean         moyenne des essais Post par souris et par mois
#   Summary_Table_<Opto|Opto_Chemo>_Post_delta        (essai 1 - essai 3) par souris et par mois
#   Summary_Table_<Opto|Opto_Chemo>_Post_slope        pente sur les essais 1-2-3 (signe inversé) par souris et par mois
#
# SENS DES SIGNES (identique pour delta et pente) : POSITIF = la valeur DIMINUE au fil des essais.
#   -> pour latency_to_reward_s, distance_to_reward_cm... : positif = la souris trouve plus vite / plus droit = MIEUX
#   -> pour un paramètre où plus grand = mieux (ex : time_spent_in_zone_percent), positif = moins bien
#   delta = essai 1 - essai 3 ; pente = - (pente de la droite valeur ~ numéro d'essai, sur les essais 1, 2, 3)
# puis Post_mean est normalisé APRÈS le calcul (4 mois = référence) :
#   ..._Post_mean_normalized
# Post_delta reste en valeurs BRUTES (normalize_delta = False) : un delta normalisé ajoute du bruit
# (référence proche de 0) et perd des souris sans donnée à 4 mois.
# Les tableaux d'origine ne sont jamais modifiés.
#
# Normalisation (comme dans la cellule de normalisation du notebook : référence par souris,
# référence = 0 -> NaN, colonnes p_value / z_ / performance_ / window_duration retirées) :
#   Post_mean  : valeur / valeur à 4 mois                         (4 mois = 1)
#   Post_delta : seulement si normalize_delta = True (désactivé par défaut) :
#                "ratio_to_ref_mean" = delta / MOYENNE à 4 mois ; "minus_ref_delta" = delta - delta à 4 mois.
##############################################################################
import os
import numpy as np
import pandas as pd

folder_path = "//10.69.168.1/crnldata/forgetting/Carla/Cheeseboard"   # <- tableaux lus et enregistrés ICI

ref_age     = "4months"           # âge de référence, tel qu'écrit dans la colonne 'age'
trial_first = 1                   # premier essai
trial_last  = 3                   # dernier essai
slope_trials     = [1, 2, 3]      # essais utilisés pour la pente
min_trials_slope = 3              # nb minimum d'essais dans la séance pour calculer la pente (3 = les trois essais)
reward_params = ['latency_to_reward_s', 'distance_to_reward_cm', 'average_speed_to_reward_cm_s']   # résumé affiché à la fin
normalize_delta   = False         # True = crée aussi Post_delta_normalized (non recommandé, voir en-tête)
delta_norm_method = "ratio_to_ref_mean"      # utilisé seulement si normalize_delta = True ("minus_ref_delta" possible)
min_trials_mean   = 1             # nb minimum d'essais pour garder un mois dans Post_mean

# experiment -> (tableau brut de départ, préfixe des tableaux créés)
experiments = {
    'Opto':       ('Summary_Table_Opto',       'Summary_Table_Opto'),
    'Opto_Chemo': ('Summary_Table_Chemo_Opto', 'Summary_Table_Opto_Chemo'),
}

# Colonnes d'identification / qualité (non moyennées, non normalisées)
keep_raw = ['mice', 'age', 'genotype', 'task_type', 'experiment', 'session_type', 'session', 'session_time',
            'trial', 'file', 'start_quadrant_loc', 'reward_location_pix', 'reward_quadrant_loc',
            'start_time', 'end_time', 'frame_rate', 'n_frames_total', 'pct_frames_detected',
            'mean_likelihood', 'reward_zone_radius_cm', 'sex', 'implant', 'virus']
mouse_attrs = ['genotype', 'sex', 'implant', 'virus']          # propriétés de la souris (constantes)
drop_prefixes = ('p_value', 'z_', 'performance_', 'window_duration')   # pas de ratio possible (comme le notebook)


def age_to_num(s):
    return pd.to_numeric(s.astype(str).str.extract(r'(\d+)')[0], errors='coerce')


def finish(tbl, session_type, count_col, attrs):
    """Ajoute les propriétés de la souris, ordonne les colonnes et trie par souris / âge."""
    tbl = tbl.reset_index().merge(attrs.reset_index(), on=['mice', 'experiment'], how='left')
    tbl['session_type'] = session_type
    tbl['_age_num'] = age_to_num(tbl['age'])
    tbl = tbl.sort_values(['mice', '_age_num']).drop(columns='_age_num').reset_index(drop=True)
    first = ['mice', 'age', 'genotype', 'experiment', 'session_type', 'sex', 'implant', 'virus', count_col]
    first = [c for c in first if c in tbl.columns]
    return tbl[first + [c for c in tbl.columns if c not in first]]


def make_mean_table(post, params, attrs):
    g = post.groupby(['mice', 'experiment', 'age'], sort=False)
    tbl = g[params].mean().join(g.size().rename('n_trials'))
    tbl = tbl[tbl['n_trials'] >= min_trials_mean]
    return finish(tbl, 'Post_mean', 'n_trials', attrs)


def make_delta_table(post, params, attrs):
    keys = ['mice', 'experiment', 'age', 'session_time']           # 1 séance Post = 1 session_time
    t = post[post['trial'].isin([trial_first, trial_last])]
    t1 = t[t['trial'] == trial_first].groupby(keys, dropna=False)[params].mean()
    t3 = t[t['trial'] == trial_last ].groupby(keys, dropna=False)[params].mean()
    d_sess = (t1 - t3).dropna(how='all')                             # essai 1 - essai 3 ; seulement les séances ayant les 2 essais
    g = d_sess.groupby(level=['mice', 'experiment', 'age'])         # moyenne des séances du même mois
    tbl = g.mean().join(g.size().rename('n_sessions'))
    return finish(tbl, 'Post_delta', 'n_sessions', attrs), d_sess.reset_index()


def make_slope_table(post, params, attrs):
    """Pente (signe inversé) de valeur ~ numéro d'essai, par séance, puis moyenne par souris et par mois."""
    keys = ['mice', 'experiment', 'age', 'session_time']
    t = post[post['trial'].isin(slope_trials)]
    rows = []
    for k, grp in t.groupby(keys, dropna=False):
        row = dict(zip(keys, k))
        for p in params:
            y = grp.groupby('trial')[p].mean().dropna()                 # 1 valeur par essai
            if len(y) >= min_trials_slope:
                row[p] = -np.polyfit(y.index.values.astype(float), y.values, 1)[0]   # signe inversé : positif = diminue
            else:
                row[p] = np.nan
        rows.append(row)
    s_sess = pd.DataFrame(rows).set_index(keys).dropna(how='all')       # seulement les séances avec assez d'essais
    g = s_sess.groupby(level=['mice', 'experiment', 'age'])             # moyenne des séances du même mois
    tbl = g.mean().join(g.size().rename('n_sessions'))
    return finish(tbl, 'Post_slope', 'n_sessions', attrs), s_sess.reset_index()


def resume_reward(tbl, label):
    cols = [c for c in reward_params if c in tbl.columns]
    if not cols:
        return
    g = tbl.groupby('genotype')[cols]
    print(f'   --- {label} : médiane par génotype (positif = diminue, ex. trouve plus vite) ---')
    print(g.median().round(2).to_string())
    print('   % de souris x mois avec valeur > 0 :')
    print(g.apply(lambda d: (d > 0).mean().mul(100).round(0)).to_string())


def aligned_ref(tbl, ref_tbl, cols):
    """Valeur de référence (à ref_age, de ref_tbl) alignée sur chaque ligne de tbl, par souris."""
    ref = ref_tbl[ref_tbl['age'] == ref_age].set_index(['mice', 'experiment'])[cols]
    ref = ref[~ref.index.duplicated()]
    ref_al = ref.reindex(pd.MultiIndex.from_frame(tbl[['mice', 'experiment']]))
    ref_al.index = tbl.index
    return ref_al


def check_ref(tbl, label):
    if not (tbl['age'] == ref_age).any():
        print(f'   /!\\ {label} : aucune ligne avec age == "{ref_age}". Ages : {sorted(tbl["age"].astype(str).unique())}')
        return
    missing = sorted(set(tbl['mice']) - set(tbl.loc[tbl['age'] == ref_age, 'mice']))
    if missing:
        print(f'   /!\\ {label} : souris sans donnée à {ref_age} (valeurs normalisées = NaN) : {missing}')


def normalize_mean(mean_tbl, params):
    norm_params = [p for p in params if not p.startswith(drop_prefixes)]
    ref = aligned_ref(mean_tbl, mean_tbl, norm_params)
    ref = ref.where(ref != 0)                                       # référence = 0 -> NaN
    out = mean_tbl.drop(columns=[p for p in params if p not in norm_params])
    out[norm_params] = mean_tbl[norm_params] / ref
    out['session_type'] = 'Post_mean'
    check_ref(mean_tbl, 'Post_mean')
    return out


def make_delta_normalized(delta_tbl, mean_tbl, params):
    norm_params = [p for p in params if not p.startswith(drop_prefixes)]
    out = delta_tbl.drop(columns=[p for p in params if p not in norm_params])
    if delta_norm_method == "ratio_to_ref_mean":
        ref = aligned_ref(delta_tbl, mean_tbl, norm_params)          # niveau moyen à 4 mois (Post_mean)
        ref = ref.where(ref != 0)
        out[norm_params] = delta_tbl[norm_params] / ref
    elif delta_norm_method == "minus_ref_delta":
        ref = aligned_ref(delta_tbl, delta_tbl, norm_params)         # delta à 4 mois
        out[norm_params] = delta_tbl[norm_params] - ref
    else:
        raise ValueError('delta_norm_method doit être "ratio_to_ref_mean" ou "minus_ref_delta"')
    check_ref(mean_tbl, 'Post_delta (référence Post_mean)')
    return out


def save_xlsx(df, name, detail=None):
    path = f'{folder_path}/{name}.xlsx'
    with pd.ExcelWriter(path) as w:
        df.to_excel(w, index=False, sheet_name='data')                # 1re feuille = le tableau (lu par R)
        if detail is not None:
            detail.to_excel(w, index=False, sheet_name='detail_par_seance')
    print(f'   {name}.xlsx  ({len(df)} lignes souris x mois)')


results = {}
for exp_name, (src_name, out_prefix) in experiments.items():
    src = f'{folder_path}/{src_name}.xlsx'
    if not os.path.exists(src):
        print(f'\n{exp_name} : {src} introuvable -> ignoré')
        continue
    print(f'\n=== {exp_name}  (source : {src_name}.xlsx) ===')
    df = pd.read_excel(src)
    post = df[df['session_type'] == 'Post'].copy()
    if post.empty:
        print('   aucune session Post -> ignoré')
        continue

    cand = [c for c in post.columns if c not in keep_raw]
    post[cand] = post[cand].apply(pd.to_numeric, errors='coerce')
    params = [c for c in cand if post[c].notna().any()]               # colonnes vides en Post écartées
    print(f'   {len(post)} essais Post, {post["mice"].nunique()} souris, {len(params)} paramètres')
    if 'sex' in post.columns and post['sex'].isna().all():
        print("   /!\\ colonne 'sex' vide : lancez d'abord la cellule 'Compléter sex / implant' du notebook v7_2")

    attr_cols = [c for c in mouse_attrs if c in post.columns]
    attrs = post.groupby(['mice', 'experiment'])[attr_cols].first()   # .first() = 1re valeur non vide

    mean_tbl            = make_mean_table(post, params, attrs)
    delta_tbl, d_detail = make_delta_table(post, params, attrs)
    slope_tbl, s_detail = make_slope_table(post, params, attrs)
    mean_norm           = normalize_mean(mean_tbl, params)
    delta_norm          = make_delta_normalized(delta_tbl, mean_tbl, params) if normalize_delta else None

    save_xlsx(mean_tbl,   f'{out_prefix}_Post_mean')
    save_xlsx(delta_tbl,  f'{out_prefix}_Post_delta', detail=d_detail)
    save_xlsx(slope_tbl,  f'{out_prefix}_Post_slope', detail=s_detail)
    save_xlsx(mean_norm,  f'{out_prefix}_Post_mean_normalized')
    if normalize_delta:
        save_xlsx(delta_norm, f'{out_prefix}_Post_delta_normalized')
    resume_reward(delta_tbl, 'delta (essai 1 - essai 3)')
    resume_reward(slope_tbl, 'pente (essais 1-2-3)')
    results[exp_name] = dict(mean=mean_tbl, delta=delta_tbl, slope=slope_tbl, mean_norm=mean_norm, delta_norm=delta_norm)

print('\nTerminé.')
